# VL-JEPA Architecture Walkthrough

This notebook walks through the Vision-Language JEPA architecture, showing how each component works.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
from src.models.vision_encoder import VisionEncoder
from src.models.language_encoder import LanguageEncoder
from src.models.predictor import Predictor
from src.models.vl_jepa import VLJEPA, default_config

## 1. Vision Encoder

The vision encoder is a ViT-Small (patch size 16, embed dim 384, 6 layers, 6 heads).
It processes image patches and returns patch embeddings + a CLS token embedding.

In [ ]:
vision_config = {"embed_dim": 384, "patch_size": 16, "image_size": 224}
vision_enc = VisionEncoder(vision_config)

dummy_image = torch.randn(1, 3, 224, 224)
patch_emb, cls_emb = vision_enc(dummy_image)

print(f"Patch embeddings shape: {patch_emb.shape}")  # [1, 196, 384]
print(f"CLS embedding shape: {cls_emb.shape}")        # [1, 384]
print(f"Number of patches: {patch_emb.shape[1]} = (224/16)^2 = 14x14")

## 2. Language Encoder

The language encoder uses a frozen BERT-base-uncased model with a projection head
to map text embeddings into the shared 384-dim space.

In [ ]:
from transformers import AutoTokenizer

lang_config = {"embed_dim": 384, "language_model": "bert-base-uncased"}
lang_enc = LanguageEncoder(lang_config)

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
tokens = tokenizer("A red ceramic coffee mug on a white background", 
                   return_tensors="pt", padding="max_length", max_length=77, truncation=True)

text_emb = lang_enc(tokens["input_ids"], tokens["attention_mask"])
print(f"Text embedding shape: {text_emb.shape}")  # [1, 384]

## 3. Block Masking

JEPA uses rectangular block masking. We mask ~50% of patches as targets
and use the remaining ~50% as context.

In [ ]:
from src.training.masking import BlockMasking

masker = BlockMasking()
mask = masker.generate_mask(num_patches_h=14, num_patches_w=14, num_targets=4, target_coverage=0.5)

print(f"Context patches: {len(mask['context_indices'])}")
print(f"Target patches: {len(mask['target_indices'])}")
print(f"Coverage: {len(mask['target_indices']) / 196 * 100:.1f}%")

## 4. Full VL-JEPA Model

The full model combines all components. During forward pass:
1. Context encoder processes unmasked patches
2. Target encoder (EMA) processes all patches (no gradient)
3. Predictor predicts target embeddings from context
4. Language encoder produces text embeddings
5. Losses: JEPA predictive loss + cross-modal alignment loss

In [ ]:
model = VLJEPA(default_config())
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# Forward pass
outputs = model(
    images=dummy_image,
    text_input_ids=tokens["input_ids"],
    text_attention_mask=tokens["attention_mask"],
    mask=mask
)

for key, val in outputs.items():
    if isinstance(val, torch.Tensor):
        print(f"{key}: {val.shape}")